### **[pandas vs Polars vs DuckDB: A Data Scientist’s Guide to Choosing the Right Tool](https://pub.towardsai.net/pandas-vs-polars-vs-duckdb-a-data-scientists-guide-to-choosing-the-right-tool-9fd08f8e5119)**

| Tool   | Backend  | Execution Model            | Best For                                        |
|--------|----------|----------------------------|-------------------------------------------------|
| pandas | C/Python | Eager, single-threaded     | Small datasets, prototyping, ML integration     |
| Polars | Rust     | Lazy/Eager, multi-threaded | Large-scale analytics, data pipelines           |
| DuckDB | C++      | SQL-first, multi-threaded  | SQL workflows, embedded analytics, file queries |

In [1]:
!pip install -qq pandas polars duckdb

In [2]:
import sys

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import polars as pl
import duckdb

IN_COLAB = "google.colab" in sys.modules

np.random.seed(42)
np.set_printoptions(threshold=sys.maxsize, suppress=True, precision=4)

pd.set_option('display.width', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.float_format', '{:.4f}'.format)

%autosave 10

Autosaving every 10 seconds


In [3]:
n_rows = 5_000_000
data = {
  "category": np.random.choice(["Electronics", "Clothing", "Food", "Books"], size=n_rows),
  "region": np.random.choice(["North", "South", "East", "West"], size=n_rows),
  "amount": np.random.rand(n_rows) * 1000,
  "quantity": np.random.randint(1, 100, size=n_rows),
}
df_pandas = pd.DataFrame(data)
df_pandas.to_csv("sales_data.csv", index=False)
print(f"Created sales_data.csv with {n_rows:,} rows")

Created sales_data.csv with 5,000,000 rows


#### **Filtering Rows**

In [4]:
df_pd = pd.read_csv("sales_data.csv")
result_bracket = df_pd[(df_pd["amount"] > 500) & (df_pd["category"] == "Electronics")]
display(result_bracket.head(10))

,category,region,amount,quantity
7,Electronics,West,662.8031,80
15,Electronics,North,826.0050,25
30,Electronics,North,766.0818,7
31,Electronics,West,772.0843,36
37,Electronics,East,527.9671,35
38,Electronics,East,500.1383,45
59,Electronics,East,913.9910,79
74,Electronics,North,935.4617,25
82,Electronics,West,676.9201,46
87,Electronics,West,837.5656,38


In [5]:
result_query = df_pd.query("amount > 500 and category == 'Electronics'")
display(result_query.head(10))

,category,region,amount,quantity
7,Electronics,West,662.8031,80
15,Electronics,North,826.0050,25
30,Electronics,North,766.0818,7
31,Electronics,West,772.0843,36
37,Electronics,East,527.9671,35
38,Electronics,East,500.1383,45
59,Electronics,East,913.9910,79
74,Electronics,North,935.4617,25
82,Electronics,West,676.9201,46
87,Electronics,West,837.5656,38


In [6]:
result_str = df_pd[df_pd["category"].str.startswith("Elec")]
display(result_str.head(10))

,category,region,amount,quantity
2,Electronics,North,450.9410,93
6,Electronics,West,475.8440,61
7,Electronics,West,662.8031,80
15,Electronics,North,826.0050,25
21,Electronics,South,292.3994,13
30,Electronics,North,766.0818,7
31,Electronics,West,772.0843,36
35,Electronics,South,305.7791,40
37,Electronics,East,527.9671,35
38,Electronics,East,500.1383,45


In [7]:
df_pl = pl.read_csv("sales_data.csv")
result_pl = df_pl.filter(
  (pl.col("amount") > 500) & (pl.col("category") == "Electronics")
)
display(result_pl.head(10))

category,region,amount,quantity
str,str,f64,i64
"""Electronics""","""West""",662.803066,80
"""Electronics""","""North""",826.004963,25
"""Electronics""","""North""",766.081832,7
"""Electronics""","""West""",772.084261,36
"""Electronics""","""East""",527.967145,35
"""Electronics""","""East""",500.138335,45
"""Electronics""","""East""",913.99102,79
"""Electronics""","""North""",935.46172,25
"""Electronics""","""West""",676.920089,46


In [8]:
result_duckdb = duckdb.sql("""
    SELECT * FROM 'sales_data.csv'
    WHERE amount > 500 AND category = 'Electronics'
""").df()
display(result_duckdb.head(10))

,category,region,amount,quantity
0,Electronics,West,662.8031,80
1,Electronics,North,826.0050,25
2,Electronics,North,766.0818,7
3,Electronics,West,772.0843,36
4,Electronics,East,527.9671,35
5,Electronics,East,500.1383,45
6,Electronics,East,913.9910,79
7,Electronics,North,935.4617,25
8,Electronics,West,676.9201,46
9,Electronics,West,837.5656,38


#### **Selecting Columns**

In [9]:
result_pd = df_pd[["category", "amount"]]
display(result_pd.head(10))

,category,amount
0,Food,516.6533
1,Books,937.3372
2,Electronics,450.9410
3,Food,674.4881
4,Food,188.8479
5,Books,976.8054
6,Electronics,475.8440
7,Electronics,662.8031
8,Food,333.9119
9,Clothing,727.9332


In [10]:
result_pl = df_pl.select(["category", "amount"])
display(result_pl.head(10))

category,amount
str,f64
"""Food""",516.653322
"""Books""",937.337226
"""Electronics""",450.941022
"""Food""",674.488081
"""Food""",188.847906
"""Books""",976.805431
"""Electronics""",475.843957
"""Electronics""",662.803066
"""Food""",333.911876


In [11]:
result_duckdb = duckdb.sql("""
    SELECT category, amount FROM 'sales_data.csv'
""").df()
display(result_duckdb.head(10))

,category,amount
0,Food,516.6533
1,Books,937.3372
2,Electronics,450.9410
3,Food,674.4881
4,Food,188.8479
5,Books,976.8054
6,Electronics,475.8440
7,Electronics,662.8031
8,Food,333.9119
9,Clothing,727.9332


#### **GroupBy Aggregation**

In [12]:
result_pd = df_pd.groupby("category").agg({
  "amount": ["sum", "mean"],
  "quantity": "sum"
})
display(result_pd)

amount           quantity
                       sum     mean       sum
category                                     
Books       624750621.9421 499.9989  62463285
Clothing    625392357.3332 500.1398  62505224
Electronics 624445295.1093 499.9382  62484265
Food        625403436.0348 499.9164  62577943

In [13]:
result_pl = df_pl.group_by("category").agg([
  pl.col("amount").sum().alias("amount_sum"),
  pl.col("amount").mean().alias("amount_mean"),
  pl.col("quantity").sum().alias("quantity_sum"),
])
display(result_pl)

category,amount_sum,amount_mean,quantity_sum
str,f64,f64,i64
"""Books""",6.2475e8,499.998897,62463285
"""Food""",6.2540e8,499.916417,62577943
"""Electronics""",6.2445e8,499.938189,62484265
"""Clothing""",6.2539e8,500.139837,62505224


In [14]:
result_duckdb = duckdb.sql("""
  SELECT
    category,
    SUM(amount) as amount_sum,
    AVG(amount) as amount_mean,
    SUM(quantity) as quantity_sum
  FROM 'sales_data.csv'
  GROUP BY category
""").df()
display(result_duckdb)

,category,amount_sum,amount_mean,quantity_sum
0,Clothing,625392357.3332,500.1398,62505224.0000
1,Electronics,624445295.1093,499.9382,62484265.0000
2,Food,625403436.0348,499.9164,62577943.0000
3,Books,624750621.9421,499.9989,62463285.0000


#### **Adding Columns**

In [15]:
result_pd = df_pd.assign(
  amount_with_tax=df_pd["amount"] * 1.1,
  high_value=df_pd["amount"] > 500
)
display(result_pd.head(10))

,category,region,amount,quantity,amount_with_tax,high_value
0,Food,South,516.6533,40,568.3187,True
1,Books,East,937.3372,45,1031.0709,True
2,Electronics,North,450.9410,93,496.0351,False
3,Food,East,674.4881,46,741.9369,True
4,Food,East,188.8479,98,207.7327,False
5,Books,South,976.8054,57,1074.4860,True
6,Electronics,West,475.8440,61,523.4284,False
7,Electronics,West,662.8031,80,729.0834,True
8,Food,East,333.9119,35,367.3031,False
9,Clothing,West,727.9332,76,800.7265,True


In [16]:
result_pl = df_pl.with_columns([
  (pl.col("amount") * 1.1).alias("amount_with_tax"),
  (pl.col("amount") > 500).alias("high_value")
])
display(result_pl.head(10))

category,region,amount,quantity,amount_with_tax,high_value
str,str,f64,i64,f64,bool
"""Food""","""South""",516.653322,40,568.318654,true
"""Books""","""East""",937.337226,45,1031.070948,true
"""Electronics""","""North""",450.941022,93,496.035124,false
"""Food""","""East""",674.488081,46,741.936889,true
"""Food""","""East""",188.847906,98,207.732697,false
"""Books""","""South""",976.805431,57,1074.485974,true
"""Electronics""","""West""",475.843957,61,523.428353,false
"""Electronics""","""West""",662.803066,80,729.083372,true
"""Food""","""East""",333.911876,35,367.303064,false


In [17]:
result_duckdb = duckdb.sql("""
  SELECT *,
    amount * 1.1 as amount_with_tax,
    amount > 500 as high_value
  FROM df_pd
""").df()
display(result_duckdb.head(10))

,category,region,amount,quantity,amount_with_tax,high_value
0,Food,South,516.6533,40,568.3187,True
1,Books,East,937.3372,45,1031.0709,True
2,Electronics,North,450.9410,93,496.0351,False
3,Food,East,674.4881,46,741.9369,True
4,Food,East,188.8479,98,207.7327,False
5,Books,South,976.8054,57,1074.4860,True
6,Electronics,West,475.8440,61,523.4284,False
7,Electronics,West,662.8031,80,729.0834,True
8,Food,East,333.9119,35,367.3031,False
9,Clothing,West,727.9332,76,800.7265,True


#### **Conditional Logic**

In [18]:
result_np = df_pd.assign(
  value_tier=np.where(
    df_pd["amount"] > 700, "high",
    np.where(df_pd["amount"] > 300, "medium", "low")
  )
)
display(result_np[["category", "amount", "value_tier"]].head(10))

,category,amount,value_tier
0,Food,516.6533,medium
1,Books,937.3372,high
2,Electronics,450.9410,medium
3,Food,674.4881,medium
4,Food,188.8479,low
5,Books,976.8054,high
6,Electronics,475.8440,medium
7,Electronics,662.8031,medium
8,Food,333.9119,medium
9,Clothing,727.9332,high


In [19]:
result_pd = df_pd.assign(
  value_tier=pd.cut(
    df_pd["amount"],
    bins=[-np.inf, 300, 700, np.inf],
    labels=["low", "medium", "high"]
  )
)
display(result_pd[["category", "amount", "value_tier"]].head(10))

,category,amount,value_tier
0,Food,516.6533,medium
1,Books,937.3372,high
2,Electronics,450.9410,medium
3,Food,674.4881,medium
4,Food,188.8479,low
5,Books,976.8054,high
6,Electronics,475.8440,medium
7,Electronics,662.8031,medium
8,Food,333.9119,medium
9,Clothing,727.9332,high


In [20]:
result = df_pd.assign(
  tier=np.where(
    (df_pd["category"] == "Electronics") & (df_pd["amount"] > 500),
    "premium", "standard"
  )
)
display(result.head(10))

,category,region,amount,quantity,tier
0,Food,South,516.6533,40,standard
1,Books,East,937.3372,45,standard
2,Electronics,North,450.9410,93,standard
3,Food,East,674.4881,46,standard
4,Food,East,188.8479,98,standard
5,Books,South,976.8054,57,standard
6,Electronics,West,475.8440,61,standard
7,Electronics,West,662.8031,80,premium
8,Food,East,333.9119,35,standard
9,Clothing,West,727.9332,76,standard


In [21]:
result_pl = df_pl.with_columns(
  pl.when(pl.col("amount") > 700).then(pl.lit("high"))
    .when(pl.col("amount") > 300).then(pl.lit("medium"))
    .otherwise(pl.lit("low"))
    .alias("value_tier")
)
display(result_pl.select(["category", "amount", "value_tier"]).head(10))

category,amount,value_tier
str,f64,str
"""Food""",516.653322,"""medium"""
"""Books""",937.337226,"""high"""
"""Electronics""",450.941022,"""medium"""
"""Food""",674.488081,"""medium"""
"""Food""",188.847906,"""low"""
"""Books""",976.805431,"""high"""
"""Electronics""",475.843957,"""medium"""
"""Electronics""",662.803066,"""medium"""
"""Food""",333.911876,"""medium"""


In [22]:
result_duckdb = duckdb.sql("""
  SELECT category, amount,
    CASE
      WHEN amount > 700 THEN 'high'
      WHEN amount > 300 THEN 'medium'
      ELSE 'low'
    END as value_tier
  FROM df_pd
""").df()
display(result_duckdb.head(10))

,category,amount,value_tier
0,Food,516.6533,medium
1,Books,937.3372,high
2,Electronics,450.9410,medium
3,Food,674.4881,medium
4,Food,188.8479,low
5,Books,976.8054,high
6,Electronics,475.8440,medium
7,Electronics,662.8031,medium
8,Food,333.9119,medium
9,Clothing,727.9332,high


#### **Window Functions**

In [23]:
result_pd = df_pd.assign(
  category_avg=df_pd.groupby("category")["amount"].transform("mean"),
  category_rank=df_pd.groupby("category")["amount"].rank(ascending=False)
)
display(result_pd[["category", "amount", "category_avg", "category_rank"]].head(10))

,category,amount,category_avg,category_rank
0,Food,516.6533,499.9164,604342.0000
1,Books,937.3372,499.9989,78423.0000
2,Electronics,450.9410,499.9382,685881.0000
3,Food,674.4881,499.9164,407088.0000
4,Food,188.8479,499.9164,1015211.0000
5,Books,976.8054,499.9989,28961.0000
6,Electronics,475.8440,499.9382,654883.0000
7,Electronics,662.8031,499.9382,421348.0000
8,Food,333.9119,499.9164,832904.0000
9,Clothing,727.9332,500.1398,340594.0000


In [24]:
result_pl = df_pl.with_columns([
  pl.col("amount").mean().over("category").alias("category_avg"),
  pl.col("amount").rank(descending=True).over("category").alias("category_rank")
])
display(result_pl.select(["category", "amount", "category_avg", "category_rank"]).head(10))

category,amount,category_avg,category_rank
str,f64,f64,f64
"""Food""",516.653322,499.916417,604342.0
"""Books""",937.337226,499.998897,78423.0
"""Electronics""",450.941022,499.938189,685881.0
"""Food""",674.488081,499.916417,407088.0
"""Food""",188.847906,499.916417,1.015211e6
"""Books""",976.805431,499.998897,28961.0
"""Electronics""",475.843957,499.938189,654883.0
"""Electronics""",662.803066,499.938189,421348.0
"""Food""",333.911876,499.916417,832904.0


In [25]:
result_duckdb = duckdb.sql("""
  SELECT category, amount,
    AVG(amount) OVER (PARTITION BY category) as category_avg,
    RANK() OVER (PARTITION BY category ORDER BY amount DESC) as category_rank
  FROM df_pd
""").df()
display(result_duckdb.head(10))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,category,amount,category_avg,category_rank
0,Clothing,999.9999,500.1398,1
1,Clothing,999.9995,500.1398,2
2,Clothing,999.9992,500.1398,3
3,Clothing,999.9982,500.1398,4
4,Clothing,999.9962,500.1398,5
5,Clothing,999.9947,500.1398,6
6,Clothing,999.9943,500.1398,7
7,Clothing,999.9935,500.1398,8
8,Clothing,999.9928,500.1398,9
9,Clothing,999.9913,500.1398,10


#### **Data Loading Performance**

In [26]:
pandas_time = %timeit -o pd.read_csv("sales_data.csv")

2.81 s ± 577 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [27]:
polars_time = %timeit -o pl.read_csv("sales_data.csv")

859 ms ± 7.68 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [28]:
duckdb_time = %timeit -o duckdb.sql("SELECT * FROM 'sales_data.csv'").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

3.15 s ± 561 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [29]:
print(f"Polars is {pandas_time.average / polars_time.average:.1f}× faster than pandas")
print(f"DuckDB is {pandas_time.average / duckdb_time.average:.1f}× faster than pandas")

Polars is 3.3× faster than pandas
DuckDB is 0.9× faster than pandas


#### **Query Optimization**

In [30]:
def pandas_query():
  return (
    pd.read_csv("sales_data.csv")
    .query('amount > 100')
    .groupby('category')['amount']
    .mean()
  )
pandas_opt_time = %timeit -o pandas_query()

3.5 s ± 483 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [31]:
def pandas_query_optimized():
  return (
    pd.read_csv("sales_data.csv", usecols=["category", "amount"])
    .query('amount > 100')
    .groupby('category')['amount']
    .mean()
  )
pandas_usecols_time = %timeit -o pandas_query_optimized()

2.67 s ± 359 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [32]:
query_pl = (
  pl.scan_csv("sales_data.csv")
  .filter(pl.col("amount") > 100)
  .group_by("category")
  .agg(pl.col("amount").mean().alias("avg_amount"))
)
# View the optimized query plan
display(query_pl.explain())

'AGGREGATE[maintain_order: false]\n  [col("amount").mean().alias("avg_amount")] BY [col("category")]\n  FROM\n  Csv SCAN [sales_data.csv]\n  PROJECT 2/4 COLUMNS\n  SELECTION: [(col("amount")) > (100.0)]\n  ESTIMATED ROWS: 4991321'

In [33]:
def polars_query():
  return (
    pl.scan_csv("sales_data.csv")
    .filter(pl.col("amount") > 100)
    .group_by("category")
    .agg(pl.col("amount").mean().alias("avg_amount"))
    .collect()
  )
polars_opt_time = %timeit -o polars_query()

1.01 s ± 266 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [34]:
def duckdb_query():
  return duckdb.sql("""
    SELECT category, AVG(amount) as avg_amount
    FROM 'sales_data.csv'
    WHERE amount > 100
    GROUP BY category
  """).df()
duckdb_opt_time = %timeit -o duckdb_query()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

1.77 s ± 401 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [35]:
print(f"Polars is {pandas_opt_time.average / polars_opt_time.average:.1f}× faster than pandas")
print(f"DuckDB is {pandas_opt_time.average / duckdb_opt_time.average:.1f}× faster than pandas")

Polars is 3.5× faster than pandas
DuckDB is 2.0× faster than pandas


#### **GroupBy Performance**

In [36]:
# Load data for fair comparison
df_pd = pd.read_csv("sales_data.csv")
df_pl = pl.read_csv("sales_data.csv")

In [37]:
def pandas_groupby():
  return df_pd.groupby("category")["amount"].mean()
pandas_groupby_time = %timeit -o pandas_groupby()

654 ms ± 140 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [38]:
def polars_groupby():
  return df_pl.group_by("category").agg(pl.col("amount").mean())
polars_groupby_time = %timeit -o polars_groupby()

203 ms ± 41.7 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [39]:
def duckdb_groupby():
  return duckdb.sql("""
    SELECT category, AVG(amount)
    FROM df_pd
    GROUP BY category
  """).df()
duckdb_groupby_time = %timeit -o duckdb_groupby()

226 ms ± 45.9 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [40]:
print(f"Polars is {pandas_groupby_time.average / polars_groupby_time.average:.1f}× faster than pandas")
print(f"DuckDB is {pandas_groupby_time.average / duckdb_groupby_time.average:.1f}× faster than pandas")

Polars is 3.2× faster than pandas
DuckDB is 2.9× faster than pandas


#### **Memory Efficiency**

In [41]:
df_pd_mem = pd.read_csv("sales_data.csv")
pandas_mem = df_pd_mem.memory_usage(deep=True).sum() / 1e3
print(f"pandas memory usage: {pandas_mem:,.0f} KB")

pandas memory usage: 627,495 KB


In [42]:
result_pl_stream = (
  pl.scan_csv("sales_data.csv")
  .group_by("category")
  .agg(pl.col("amount").mean())
  .collect(streaming=True)
)
polars_mem = result_pl_stream.estimated_size() / 1e3
print(f"Polars result memory: {polars_mem:.2f} KB")

Polars result memory: 0.06 KB


In [43]:
(
  pl.scan_csv("sales_data.csv")
  .filter(pl.col("amount") > 500)
  .sink_parquet("filtered_sales.parquet")
)

In [44]:
# Configure memory limit and temp directory
duckdb.sql("SET memory_limit = '500MB'")
duckdb.sql("SET temp_directory = '/tmp/duckdb_temp'")
# DuckDB handles larger-than-RAM automatically
result_duckdb_mem = duckdb.sql("""
    SELECT category, AVG(amount) as avg_amount
    FROM 'sales_data.csv'
    GROUP BY category
""").df()
duckdb_mem = result_duckdb_mem.memory_usage(deep=True).sum() / 1e3
print(f"DuckDB result memory: {duckdb_mem:.2f} KB")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

DuckDB result memory: 0.39 KB


In [45]:
print(f"pandas: {pandas_mem:,.0f} KB (full dataset)")
print(f"Polars: {polars_mem:.2f} KB (result only)")
print(f"DuckDB: {duckdb_mem:.2f} KB (result only)")
print(f"\nPolars uses {pandas_mem / polars_mem:,.0f}× less memory than pandas")
print(f"DuckDB uses {pandas_mem / duckdb_mem:,.0f}× less memory than pandas")

pandas: 627,495 KB (full dataset)
Polars: 0.06 KB (result only)
DuckDB: 0.39 KB (result only)

Polars uses 10,458,252× less memory than pandas
DuckDB uses 1,617,255× less memory than pandas


#### **Join Operations**

In [46]:
# Create orders table (1M rows)
orders_pd = pd.DataFrame({
  "order_id": range(1_000_000),
  "customer_id": np.random.randint(1, 100_000, size=1_000_000),
  "amount": np.random.rand(1_000_000) * 500
})
# Create customers table (100K rows)
customers_pd = pd.DataFrame({
  "customer_id": range(100_000),
  "region": np.random.choice(["North", "South", "East", "West"], size=100_000)
})
# Convert to Polars
orders_pl = pl.from_pandas(orders_pd)
customers_pl = pl.from_pandas(customers_pd)

In [47]:
def pandas_join():
  return orders_pd.merge(customers_pd, on="customer_id", how="left")
pandas_join_time = %timeit -o pandas_join()

148 ms ± 25.8 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [48]:
def polars_join():
  return orders_pl.join(customers_pl, on="customer_id", how="left")
polars_join_time = %timeit -o polars_join()

35.9 ms ± 1.07 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [49]:
def duckdb_join():
  return duckdb.sql("""
    SELECT o.*, c.region
    FROM orders_pd o
    LEFT JOIN customers_pd c ON o.customer_id = c.customer_id
  """).df()
duckdb_join_time = %timeit -o duckdb_join()

175 ms ± 33.9 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [50]:
print(f"Polars is {pandas_join_time.average / polars_join_time.average:.1f}× faster than pandas")
print(f"DuckDB is {pandas_join_time.average / duckdb_join_time.average:.1f}× faster than pandas")

Polars is 4.1× faster than pandas
DuckDB is 0.8× faster than pandas


#### **Interoperability**

In [51]:
df = pd.DataFrame({
  "product": ["A", "B", "C"],
  "sales": [100, 200, 150]
})
# DuckDB queries pandas DataFrames by variable name
result = duckdb.sql("SELECT * FROM df WHERE sales > 120").df()
display(result)

,product,sales
0,B,200
1,C,150


In [52]:
df_polars = pl.DataFrame({
  "feature1": [1, 2, 3],
  "feature2": [4, 5, 6],
  "target": [0, 1, 0]
})
# Convert to pandas for scikit-learn
df_pandas = df_polars.to_pandas()
print(type(df_pandas))

<class 'pandas.core.frame.DataFrame'>


In [53]:
result = duckdb.sql("""
  SELECT category, SUM(amount) as total
  FROM 'sales_data.csv'
  GROUP BY category
""").pl()
print(type(result))
display(result)

<class 'polars.dataframe.frame.DataFrame'>


category,total
str,f64
"""Electronics""",6.2445e8
"""Food""",6.2540e8
"""Books""",6.2475e8
"""Clothing""",6.2539e8


#### **Combined Pipeline Example**

In [54]:
# Step 1: DuckDB for initial SQL query
aggregated = duckdb.sql("""
  SELECT category, region,
        SUM(amount) as total_amount,
        COUNT(*) as order_count
  FROM 'sales_data.csv'
  GROUP BY category, region
""").pl()

# Step 2: Polars for additional transformations
enriched = (
  aggregated
  .with_columns([
    (pl.col("total_amount") / pl.col("order_count")).alias("avg_order_value"),
    pl.col("category").str.to_uppercase().alias("category_upper")
  ])
  .filter(pl.col("order_count") > 100000)
)

# Step 3: Convert to pandas for visualization or ML
final_df = enriched.to_pandas()
display(final_df.head())

,category,region,total_amount,order_count,avg_order_value,category_upper
0,Clothing,South,156341215.8948,312610,500.1159,CLOTHING
1,Food,West,156099417.0304,312662,499.2593,FOOD
2,Electronics,East,155675840.0576,311518,499.7330,ELECTRONICS
3,Electronics,North,156612460.9710,313410,499.7047,ELECTRONICS
4,Food,South,156559604.9128,312799,500.5118,FOOD
